In [1]:
import numpy as np
from pymatreader import read_mat
import matplotlib.pyplot as plt
import matplotlib.patches as patches

%matplotlib inline
plt.rcParams['figure.dpi'] = 150
plt.rcParams['font.size'] = 11

DATA_DIR = '../data/'
WERNLE_DIR = DATA_DIR + 'wernle/'
GRID_DIR = DATA_DIR + 'grids/'

### Functions

In [8]:
def plot_maps(J, U, C, mC, P, lxy, figsize=(14, 8)):
    plt_nr = 2
    plt_nc = 4
    n = int(np.sqrt(U.shape[0]))
    Z = np.column_stack([J[:, 0], U[:, 0], C[:, 0], mC, J[:, 1], U[:, 1], C[:, 1]])
    K = Z.shape[1]
    sub_plots = [1, 2, 3, 4, 5, 6, 7]
    fs = 14

    fig = plt.figure(figsize=figsize)

    for i in range(1, K + 1):
        ax = fig.add_subplot(plt_nr, plt_nc, sub_plots[i - 1])

        map_ = Z[:, i - 1].reshape(n, n, order='C')  # row-major = MATLAB's imagesc(map')

        if i == 3 or i == 7:
            im = ax.imshow(map_, origin='lower', cmap='jet', vmin=-0.3, vmax=1, aspect='equal')
        elif i == 4:
            im = ax.imshow(map_, origin='lower', cmap='jet', vmin=0.4, vmax=0.8, aspect='equal')
        else:
            im = ax.imshow(map_, origin='lower', cmap='jet', aspect='equal')

        ax.set_xticks([])
        ax.set_yticks([])
        for spine in ax.spines.values():
            spine.set_visible(True)

        if i in [3, 7]:
            hc = plt.colorbar(im, ax=ax)
            hc.set_ticks([-0.3, 1])
            hc.ax.tick_params(labelsize=fs)

        if i == 4:
            hc = plt.colorbar(im, ax=ax)
            hc.set_ticks([0.4, 0.8])
            hc.ax.tick_params(labelsize=fs)

        if i == 1 or i == 5:
            config = {
                'linewidth': 3,
                'linecolor': 'w',
                'text_states': 0,
                'start': [],
                'last': [],
                'colorful': np.zeros(P.shape[0]),
                'show_terminals': 0,
                'add_grid': 0,
            }
            plot_walls(ax, P, lxy, config)

    plt.tight_layout()
    plt.show()


def plot_walls(ax, P, lxy, config=None):
    default_config = {
        'linewidth': 1,
        'text_states': 0,
        'start': [],
        'last': [],
        'colorful': np.zeros(P.shape[0]),
        'linecolor': 'k',
        'show_terminals': 0,
        'add_grid': 0,
    }
    cfg = {**default_config, **(config or {})}

    W = 1 - (P > 0).astype(float)
    terminals = np.where(np.diag(P) != 0)[0]

    lxy_arr = np.array(lxy)
    states = lxy_arr[:, 0].astype(int) - 1  # convert 1-based to 0-based
    rr = lxy_arr[:, 1]
    cc = lxy_arr[:, 2]
    n_states = len(states)

    ptr_down  = np.zeros(n_states, dtype=int)
    ptr_right = np.zeros(n_states, dtype=int)

    for idx_i in range(n_states):
        i = states[idx_i]
        xyi = lxy_arr[idx_i, 1:3]

        # Check right neighbor
        xyir = xyi + np.array([1, 0])
        j_idx = np.where((lxy_arr[:, 1] == xyir[0]) & (lxy_arr[:, 2] == xyir[1]))[0]
        if len(j_idx) > 0:
            j = states[j_idx[0]]
            if W[i, j] != 0:
                ptr_down[idx_i] = 1

        # Check upper neighbor
        xyir = xyi + np.array([0, 1])
        j_idx = np.where((lxy_arr[:, 1] == xyir[0]) & (lxy_arr[:, 2] == xyir[1]))[0]
        if len(j_idx) > 0:
            j = states[j_idx[0]]
            if W[i, j] != 0:
                ptr_right[idx_i] = 1

    constant = 0.5
    colorful = np.asarray(cfg['colorful'])

    for idx_i in range(n_states):
        i = states[idx_i]

        if ptr_right[idx_i] == 1:
            ax.plot([cc[idx_i] + constant, cc[idx_i] + constant],
                    [rr[idx_i] - constant, rr[idx_i] + constant],
                    linewidth=cfg['linewidth'], color=cfg['linecolor'])

        if ptr_down[idx_i] == 1:
            ax.plot([cc[idx_i] - constant, cc[idx_i] + constant],
                    [rr[idx_i] + constant, rr[idx_i] + constant],
                    linewidth=cfg['linewidth'], color=cfg['linecolor'])

        if cfg['add_grid']:
            ax.plot([cc[idx_i] - constant, cc[idx_i] + constant],
                    [rr[idx_i] + constant, rr[idx_i] + constant],
                    linewidth=0.5, color=[0.5, 0.5, 0.5])
            ax.plot([cc[idx_i] + constant, cc[idx_i] + constant],
                    [rr[idx_i] - constant, rr[idx_i] + constant],
                    linewidth=0.5, color=[0.5, 0.5, 0.5])

        if cfg['text_states'] and i < 1000:
            ax.text(cc[idx_i], rr[idx_i], f'{i + 1:3d}', ha='center')

        if colorful[i]:
            ax.add_patch(patches.Rectangle(
                (cc[idx_i] - constant, rr[idx_i] - constant), 1, 1,
                facecolor='r', edgecolor='r'))

        if cfg['show_terminals'] and np.any(terminals == i):
            ax.add_patch(patches.Rectangle(
                (cc[idx_i] - constant, rr[idx_i] - constant), 1, 1,
                facecolor='r', edgecolor='r'))

    start = cfg['start']
    if hasattr(start, '__len__') and len(start) > 0 or (not hasattr(start, '__len__') and start):
        idx_i = np.where(states == int(start) - 1)[0][0]
        ax.add_patch(patches.Rectangle(
            (cc[idx_i] - constant, rr[idx_i] - constant), 1, 1,
            facecolor='b', edgecolor='b'))

    last = cfg['last']
    if hasattr(last, '__len__') and len(last) > 0 or (not hasattr(last, '__len__') and last):
        idx_i = np.where(states == int(last) - 1)[0][0]
        ax.add_patch(patches.Rectangle(
            (cc[idx_i] - constant, rr[idx_i] - constant), 1, 1,
            facecolor='r', edgecolor='r'))

    ax.set_xlim([-constant + min(cc), constant + max(cc)])
    ax.set_ylim([-constant + min(rr), constant + max(rr)])
    ax.set_xticks([])
    ax.set_yticks([])
    for spine in ax.spines.values():
        spine.set_visible(True)

### Main

In [20]:
wernle = read_mat(GRID_DIR+'wernle.mat')
open_basis = read_mat(GRID_DIR+'open_basis.mat')

In [22]:
open_basis.keys()

dict_keys(['__header__', '__version__', '__globals__', 'arragmentPC', 'centers', 'fname', 'initial_J', 'maxx', 'maxy', 'NmEC', 'Nx', 'Ny', 'paint', 'PcSize', 'placeCellType', 'rng_seed', 'simdur', 'step_cost', 'Resolution', 'N1', 'epsilon', 'delta', 'maxWeight', 'minx', 'miny', 'vel', 'angular', 'psiSat', 'meanZaro', 'b1', 'b2', 'Arc', 'output', 'NonNegativity', 'saveInput', 'placeCenters', 'DisksData', 'alpha', 'J', 'J_history', 'D', 'PCAEvec', 'Cov', 'outputMap', 'autocorrMap', 'Gridness60', 'Gridness90', 'minAngles', 'GridSpacing', 'lxy'])

In [10]:
corr_map = wernle['corr_map']
exemplar = wernle['exemplar']

In [11]:
task = exemplar['task']
model = exemplar['model']
C = exemplar['C']

In [23]:
lxy = task['lxy']
P0 = task['P0']
U = model['U']
# J = task['J']
J = open_basis['J']
mC = np.nanmedian(corr_map, axis=1)  # shape: (A, C)  — NumPy drops dim 1
mC = np.mean(mC, axis=1)             # shape: (A,)    — now axis=1, not 2

In [25]:
J.shape

(625, 50)

In [ ]:
# --- Entry point (equivalent to the top-level script) ---
idx = [3, 18]  # MATLAB [4 19] → Python 0-indexed [3, 18]

fig = plt.figure()
manager = plt.get_current_fig_manager()
# Approximate normalized figure size [0 0 .7 .4] using figsize
fig.set_size_inches(14, 8)  # adjust to taste

plot_maps(J[:, idx], U[:, idx], C[:, idx], mC, P0, lxy)

ValueError: all the input array dimensions except for the concatenation axis must match exactly, but along dimension 0, the array at index 0 has size 625 and the array at index 1 has size 900

<Figure size 2100x1200 with 0 Axes>

In [19]:
C.shape

(900, 50)